# Lab 2  Messy Data & Leakage-Safe Features

**DSA 8401 Applied Machine Learning · Chapter 2**

This notebook works the four Lab 2 tasks on the synthetic mobile-money dump `mobile_money_statements.csv` using the patterns from Chapter 2

1. **Audit**  profile missingness / duplicates / validity violations; classify each
   incomplete column MCAR / MAR / MNAR 
2. **Clean & aggregate**  parse amounts and dates
     - resolve entities to a canonical `customer_id` 
     -  build customer-level RFM, ratio and cyclical features 
3. **Hunt the leak**  correlation screen (Listing 2.11) + a lineage argument to find the two planted leaks.
4. **Pipeline**  one `ColumnTransformer` `Pipeline`, group-aware CV, ROC–AUC with and without the leaks 



In [1]:
import re, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")



In [2]:
# The scoring timestamp: every feature is computed over data strictly BEFORE this
SCORING_TS = pd.Timestamp("2026-08-01 00:00:00")



In [4]:
raw = pd.read_csv(r"C:\Users\HP\OneDrive\Documents\Applied Machine Learning\AML_Book-Data\Data\mobile_money_statements.csv")
print("raw shape:", raw.shape)
raw.head(3).T

raw shape: (183600, 18)


,0,1,2
txn_id,TXN0102856,TXN0155490,TXN0019434
msisdn,254700004022,254700013612,254700005862
reg_id,NID101089,NID103658,nid101584
account_name,Joseph Mwangi,Rehema Ndlovu,Mary Mutua
region,Kampala,Mombasa,Arusha
segment,retail,retail,retail
txn_time,2026-06-30 14:24:19,"Jan 29, 2026 01:17 PM","Sep 16, 2025 12:55 AM"
txn_type,send,send,receive
amount,"1,329/- Dr",289/- Dr,"1,571/-"
gps_lat,NaN,1.7747,-3.74336


In [5]:
raw.head(10)

,txn_id,msisdn,reg_id,account_name,region,segment,txn_time,txn_type,amount,gps_lat,gps_lon,agent_id,device_id,counterparty,balance_after,manual_review_score,settlement_status,is_fraud
0,TXN0102856,254700004022,NID101089,Joseph Mwangi,Kampala,retail,2026-06-30 14:24:19,send,"1,329/- Dr",NaN,NaN,AG1014,DV978723,CP6324,8123.0,0.061,settled,0
1,TXN0155490,254700013612,NID103658,Rehema Ndlovu,Mombasa,retail,"Jan 29, 2026 01:17 PM",send,289/- Dr,1.77470,34.70915,AG1084,DV616667,CP2831,6322.0,0.000,settled,0
2,TXN0019434,254700005862,nid101584,Mary Mutua,Arusha,retail,"Sep 16, 2025 12:55 AM",receive,"1,571/-",-3.74336,33.67018,AG1091,DV637734,CP167,15237.0,0.912,reversed,1
3,TXN0049426,254700009223,NID102493,Joseph Okoth,Dar es Salaam,retail,07/04/2026 11:21,send,599/- Dr,NaN,NaN,AG1047,DV666470,CP3382,3322.0,0.095,settled,0
4,TXN0013816,254700014760,nid103963,B. Ssentongo,Mwanza,retail,20/12/2025 11:05,cashout,"(1,307/-)",0.08284,29.02835,NaN,NaN,CP8428,2168.0,0.085,settled,0
5,TXN0068766,254700005080,NID101375,Grace Mutua,Arusha,retail,2026-06-21 12:00:08,send,"2,128/- Dr",-0.28486,33.04071,NaN,DV322595,CP6317,3202.0,0.066,settled,0
6,TXN0006514,254700010321,NID102782,SSENTONGO GRACE,Mwanza,retail,24/07/2026 04:01,bill,"-2,136/-",-3.07853,33.77321,NaN,DV492573,CP7821,4046.0,0.008,settled,0
7,TXN0115001,254700007185,NID101945,Amina Ali,Mombasa,retail,"May 25, 2026 12:17 PM",cashin,"1,026/-",NaN,NaN,AG1026,DV104452,CP6532,3226.0,0.172,settled,0
8,TXN0067557,254700011799,nid103172,G. Kamau,Mombasa,merchant,22/10/2025 22:44,cashin,"1,099/-",2.62258,33.90121,AG1099,DV674871,CP5021,6475.0,0.069,settled,0
9,TXN0086664,254700005680,NID101533,Aisha Ali,Nairobi,retail,2025-09-02 07:48:24,receive,951/-,NaN,NaN,AG1338,DV107852,CP609,4576.0,0.090,pending,0


In [6]:
# eye balling

raw.shape

(183600, 18)

In [7]:
raw.columns

Index(['txn_id', 'msisdn', 'reg_id', 'account_name', 'region', 'segment',
       'txn_time', 'txn_type', 'amount', 'gps_lat', 'gps_lon', 'agent_id',
       'device_id', 'counterparty', 'balance_after', 'manual_review_score',
       'settlement_status', 'is_fraud'],
      dtype='object')

In [8]:
raw.dtypes

txn_id                  object
msisdn                   int64
reg_id                  object
account_name            object
region                  object
segment                 object
txn_time                object
txn_type                object
amount                  object
gps_lat                float64
gps_lon                float64
agent_id                object
device_id               object
counterparty            object
balance_after          float64
manual_review_score    float64
settlement_status       object
is_fraud                 int64
dtype: object

In [9]:
raw.select_dtypes(include="object")

,txn_id,reg_id,account_name,region,segment,txn_time,txn_type,amount,agent_id,device_id,counterparty,settlement_status
0,TXN0102856,NID101089,Joseph Mwangi,Kampala,retail,2026-06-30 14:24:19,send,"1,329/- Dr",AG1014,DV978723,CP6324,settled
1,TXN0155490,NID103658,Rehema Ndlovu,Mombasa,retail,"Jan 29, 2026 01:17 PM",send,289/- Dr,AG1084,DV616667,CP2831,settled
2,TXN0019434,nid101584,Mary Mutua,Arusha,retail,"Sep 16, 2025 12:55 AM",receive,"1,571/-",AG1091,DV637734,CP167,reversed
3,TXN0049426,NID102493,Joseph Okoth,Dar es Salaam,retail,07/04/2026 11:21,send,599/- Dr,AG1047,DV666470,CP3382,settled
4,TXN0013816,nid103963,B. Ssentongo,Mwanza,retail,20/12/2025 11:05,cashout,"(1,307/-)",NaN,NaN,CP8428,settled
...,...,...,...,...,...,...,...,...,...,...,...,...
183595,TXN0066455,NID102893,"Mutua, Grace",Dar es Salaam,retail,27/11/2025 18:15,airtime,(52/-),AG1191,DV964052,CP208,settled
183596,TXN0053459,nid101960,OSEI GRACE,Arusha,retail,"Jul 18, 2026 07:03 AM",send,"(1,003/-)",NaN,DV963852,CP6035,settled
183597,TXN0010742,NID100470,OTIENO FATUMA,Nakuru,retail,05/02/2026 05:33,receive,920/-,AG1081,DV731612,CP8535,settled
183598,TXN0049689,NID100206,Z. Ssentongo,Mombasa,retail,24/03/2026 01:18,bill,"(2,848/-)",AG1163,DV746921,CP8162,reversed


In [10]:
raw['region'].unique()

array(['Kampala', 'Mombasa', 'Arusha', 'Dar es Salaam', 'Mwanza',
       'Nairobi', 'Eldoret', 'Kigali', 'Nakuru', 'Jinja'], dtype=object)

## 1. Audit(Missing  Nos)

Three cheap diagnostics :

- **quantify** the missing fraction per column, 
- **test** whether missingness is informative,
- **inspect co-missingness**. 

We also check duplicates and the validity rules the dump violates.

In [11]:
raw.isna().sum()/raw.shape[0]

txn_id                 0.000000
msisdn                 0.000000
reg_id                 0.000000
account_name           0.000000
region                 0.000000
segment                0.000000
txn_time               0.000000
txn_type               0.000000
amount                 0.000000
gps_lat                0.362108
gps_lon                0.362108
agent_id               0.136939
device_id              0.059129
counterparty           0.000000
balance_after          0.000000
manual_review_score    0.000000
settlement_status      0.000000
is_fraud               0.000000
dtype: float64

In [12]:
# 1a. Missingness fraction per column (Listing 2.1, step 1) ---


miss = raw.isna().mean().sort_values(ascending=False)

print("Missing fraction per column:")

print(miss[miss > 0].round(4).to_string())

Missing fraction per column:
gps_lon      0.3621
gps_lat      0.3621
agent_id     0.1369
device_id    0.0591


In [13]:
# 1b. Duplicates ---
# Exact row duplicates (client retries / log merges) -> drop_duplicates dispatches them.


print("exact duplicate rows :", int(raw.duplicated().sum()))

print("duplicate txn_id      :", int(raw['txn_id'].duplicated().sum()))


# txn_id repeats exactly on the duplicate rows they are true row copies, not new events.

exact duplicate rows : 3600
duplicate txn_id      : 3600


In [14]:
# 1c. Validity-rule violations ---

# amount is stored as a STRING with thousands separators, a "/-" suffix, and three
# different debit encodings; it will not cast to a number as-is.

print("amount dtype:", raw['amount'].dtype)

print("amount samples:", raw['amount'].dropna().sample(6, random_state=1).tolist())

# txn_time mixes three formats -> a single to_datetime(format=...) cannot parse all rows.
print("txn_time samples:", raw['txn_time'].sample(6, random_state=2).tolist())

amount dtype: object
amount samples: ['71/- Dr', '4,742/- Dr', '1,701/-', '(227/-)', '(165/-)', '(51/-)']
txn_time samples: ['29/06/2026 04:49', '2025-09-09 05:09:38', 'Mar 20, 2026 02:09 PM', 'May 04, 2025 08:43 PM', '2026-03-30 12:00:04', '03/08/2025 21:49']


In [15]:
raw["gps_lat"].notna()

0         False
1          True
2          True
3         False
4          True
          ...  
183595    False
183596    False
183597     True
183598     True
183599     True
Name: gps_lat, Length: 183600, dtype: bool

In [16]:
raw.loc[raw["gps_lat"].notna(), "is_fraud"].mean()

np.float64(0.08387339156569926)

In [17]:
# -1d. Is missingness informative? (the MNAR screen)

# Compare the target rate across the missing indicator for each incomplete column.

for col in ["gps_lat", "agent_id", "device_id"]:

    # Fraud rate when the value is present
    present_rate = raw.loc[raw[col].notna(), "is_fraud"].mean()

    # Fraud rate when the value is missing
    missing_rate = raw.loc[raw[col].isna(), "is_fraud"].mean()

    print(f"\n{col}")
    print(f"  Present : {present_rate:.4f}")
    print(f"  Missing : {missing_rate:.4f}")


gps_lat
  Present : 0.0839
  Missing : 0.0843

agent_id
  Present : 0.0841
  Missing : 0.0839

device_id
  Present : 0.0840
  Missing : 0.0838


In [18]:

# 1e. Can OBSERVED columns explain the missingness? (MAR test) ---

# agent_id: missing fraction by region.


# Percentage of missing agent_id values in each region
print("Missing agent_id by region")
print(
    raw.groupby("region")["agent_id"]
       .apply(lambda x: x.isna().mean())
       .round(3)
)

# Percentage of missing gps_lat values in each region
print("\nMissing gps_lat by region")
print(
    raw.groupby("region")["gps_lat"]
       .apply(lambda x: x.isna().mean())
       .round(3)
)

# Percentage of missing device_id values in each region
print("\nMissing device_id by region")
print(
    raw.groupby("region")["device_id"]
       .apply(lambda x: x.isna().mean())
       .round(3)
)

Missing agent_id by region
region
Arusha           0.452
Dar es Salaam    0.000
Eldoret          0.000
Jinja            0.451
Kampala          0.000
Kigali           0.000
Mombasa          0.000
Mwanza           0.448
Nairobi          0.000
Nakuru           0.000
Name: agent_id, dtype: float64

Missing gps_lat by region
region
Arusha           0.360
Dar es Salaam    0.362
Eldoret          0.357
Jinja            0.365
Kampala          0.365
Kigali           0.367
Mombasa          0.354
Mwanza           0.364
Nairobi          0.360
Nakuru           0.366
Name: gps_lat, dtype: float64

Missing device_id by region
region
Arusha           0.060
Dar es Salaam    0.059
Eldoret          0.058
Jinja            0.056
Kampala          0.061
Kigali           0.061
Mombasa          0.060
Mwanza           0.058
Nairobi          0.059
Nakuru           0.058
Name: device_id, dtype: float64


### Step 1  findings and MCAR / MAR / MNAR classification

- MAR -  The missing value depends on another variable that you already have. , yes we can explain
- MNAR-  Missing because of the value itself ,No we cant
- MCAR  - Completely random, No we cant The missing values happened purely by chance.


**Duplicates.** Exactly **3,600** rows are exact copies (the same `txn_id` repeats). They must be dropped *before* any split, or the same event lands in train and test( cause leakage)

**Validity violations.** `amount` is a string (`"1,500/-"`, `"(227/-)"`, `"4,742/- Dr"`) three different debit encodings; `txn_time` mixes ISO, `dd/mm/yyyy`, and `Mon dd, yyyy` formats. Both need parsing before use.

**Missingness (evidence  mechanism).** Recall Definitions 
 - MCAR = missingness independent of everything
 - MAR = fully explained by *observed* columns; 
 - MNAR = depends on the *unobserved value itself*.


| Column | Missing | Evidence | Class |
|---|---|---|---|
| `agent_id` | ~14% | Missing fraction is ~45% in **Mwanza / Jinja / Arusha** and ~0% everywhere else — missingness is **fully explained by the observed `region`** column (a legacy logger in those regions). | **MAR** |
| `device_id` | ~6% | Missing fraction is flat across region/type and the missing-indicator does **not** move the fraud rate; there is no mechanism tying it to any value. Uniform, uninformative dropout. | **MCAR** |
| `gps_lat` / `gps_lon` | ~36% | Flat across observed columns too (so **not** MAR-explainable), but network/GPS coverage is worse in remote locations, i.e. the probability a coordinate is missing **depends on the coordinate itself**. | **MNAR** |


## 2. Clean and aggregate

- Deduplicate, 
- parse amounts and dates, 
- resolve entities to a canonical `customer_id`
- build customer-level RFM / ratio / cyclical features aligned to `SCORING_TS` (Eq. 2.1, Table 2.2, Listing 2.9).

In [19]:
#- 2a. Exact deduplication 
df = raw.drop_duplicates().reset_index(drop=True)
print("after dedup:", df.shape)

after dedup: (180000, 18)


In [20]:
raw[['amount','txn_id']].sample(30)

,amount,txn_id
172846,"-6,654/-",TXN0144844
110776,(36/-),TXN0121107
8819,"8,968/-",TXN0054971
158723,(960/-),TXN0026277
2776,-197/-,TXN0011760
131695,90/- Dr,TXN0067302
62295,"1,025/- Dr",TXN0057101
94350,"-4,968/-",TXN0080203
20503,(157/-),TXN0052367
11693,"-5,864/-",TXN0057252


In [21]:

# -2b. Parse the string amount into a signed numeric amount ---

# debit if it starts with '-', is wrapped in (), or is tagged ' Dr'; credit otherwise.

def parse_amount(s):
    
    s = str(s).strip()
    
    neg = s.startswith("(") or s.startswith("-") or "Dr" in s
    
    digits = re.sub(r"[^0-9]", "", s)          # strip commas, '/-', parens, 'Dr', sign
    
    if digits == "":
        
        return np.nan
    
    v = float(digits)
    
    return -v if neg else v



In [22]:
df["amount_signed"]    = df["amount"].map(parse_amount)  

df[['amount','amount_signed']].sample(30)

,amount,amount_signed
107149,"-2,811/-",-2811.0
39370,"1,636/- Dr",-1636.0
27196,"4,490/-",4490.0
104568,(462/-),-462.0
140213,"(1,325/-)",-1325.0
139237,"1,787/- Dr",-1787.0
9053,"(1,088/-)",-1088.0
42316,"1,875/-",1875.0
103808,147/- Dr,-147.0
59845,416/- Dr,-416.0


In [23]:
df["amount_signed"]    = df["amount"].map(parse_amount)  
df["amount_abs"]       = df["amount_signed"].abs()



print("amount parse failures:", int(df["amount_signed"].isna().sum()))
df[["amount", "amount_signed"]].head(10)

amount parse failures: 0


,amount,amount_signed
0,"1,329/- Dr",-1329.0
1,289/- Dr,-289.0
2,"1,571/-",1571.0
3,599/- Dr,-599.0
4,"(1,307/-)",-1307.0
5,"2,128/- Dr",-2128.0
6,"-2,136/-",-2136.0
7,"1,026/-",1026.0
8,"1,099/-",1099.0
9,951/-,951.0


In [24]:
df[['txn_id','txn_time']]

,txn_id,txn_time
0,TXN0102856,2026-06-30 14:24:19
1,TXN0155490,"Jan 29, 2026 01:17 PM"
2,TXN0019434,"Sep 16, 2025 12:55 AM"
3,TXN0049426,07/04/2026 11:21
4,TXN0013816,20/12/2025 11:05
...,...,...
179995,TXN0066455,27/11/2025 18:15
179996,TXN0053459,"Jul 18, 2026 07:03 AM"
179997,TXN0010742,05/02/2026 05:33
179998,TXN0049689,24/03/2026 01:18


In [25]:
s = df["txn_time"].astype(str)
s

0           2026-06-30 14:24:19
1         Jan 29, 2026 01:17 PM
2         Sep 16, 2025 12:55 AM
3              07/04/2026 11:21
4              20/12/2025 11:05
                  ...          
179995         27/11/2025 18:15
179996    Jul 18, 2026 07:03 AM
179997         05/02/2026 05:33
179998         24/03/2026 01:18
179999         24/06/2025 21:13
Name: txn_time, Length: 180000, dtype: object

In [26]:
ts = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")
ts

0        2026-06-30 14:24:19
1                        NaT
2                        NaT
3                        NaT
4                        NaT
                 ...        
179995                   NaT
179996                   NaT
179997                   NaT
179998                   NaT
179999                   NaT
Name: txn_time, Length: 180000, dtype: datetime64[ns]

In [27]:
ts.isna()

0         False
1          True
2          True
3          True
4          True
          ...  
179995     True
179996     True
179997     True
179998     True
179999     True
Name: txn_time, Length: 180000, dtype: bool

In [28]:


# Format 2: 15/01/2025 14:30
mask = ts.isna()
ts.loc[mask] = pd.to_datetime(
                s[mask],
                format="%d/%m/%Y %H:%M",
                errors="coerce"
            )

In [29]:

# Convert the transaction time column to text
s = df["txn_time"].astype(str)

# Try parsing the dates using different formats

# Format 1: 2025-01-15 14:30:00

ts = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")

# Format 2: 15/01/2025 14:30
mask = ts.isna()
ts.loc[mask] = pd.to_datetime(
                s[mask],
                format="%d/%m/%Y %H:%M",
                errors="coerce"
            )

# Format 3: Jan 15, 2025 02:30 PM
mask = ts.isna()
ts.loc[mask] = pd.to_datetime(
                    s[mask],
                    format="%b %d, %Y %I:%M %p",
                    errors="coerce"
                )

# Save the parsed dates
df["ts"] = ts

# Check how many dates could not be parsed
print("Number of invalid dates:", df["ts"].isna().sum())

# Check that all transactions happened before the scoring date
print("All transactions before scoring date:",
      (df["ts"] < SCORING_TS).all())

Number of invalid dates: 0
All transactions before scoring date: True


In [30]:
df[['txn_time','ts']].dtypes

txn_time            object
ts          datetime64[ns]
dtype: object

In [31]:
df["reg_id"].nunique()

11722

In [32]:
df['reg_id'].unique()

array(['NID101089', 'NID103658', 'nid101584', ..., 'NID101841',
       '  NID100165 ', '  NID102518 '], shape=(11722,), dtype=object)

In [33]:
# - 2d. Entity resolution canonical customer_id 


# The msisdn (SIM) is NOT the identity: multi-SIM customers own several. The KYC
# registration id (reg_id) is the linking key; every SIM a person registers shares it.
# It is lightly messy (case / whitespace), so canonicalise before grouping.


# Create a clean customer ID
df["customer_id"] = (
                df["reg_id"]
                .astype(str)      # Convert to text
                .str.strip()      # Remove spaces
                .str.upper()      # Convert to uppercase
            )

# Count unique values
print("Unique SIM numbers (msisdn):", df["msisdn"].nunique())
print("Unique raw registration IDs:", df["reg_id"].nunique())
print("Unique cleaned customer IDs:", df["customer_id"].nunique())

#   keying on msisdn (or raw reg_id) would badly over-count customers and scatter a
#   person's SIMs across CV folds (leakage cause 5). We group on customer_id instead.

Unique SIM numbers (msisdn): 4825
Unique raw registration IDs: 11722
Unique cleaned customer IDs: 3991


In [34]:
# The scoring timestamp: every feature is computed over data strictly BEFORE this
SCORING_TS = pd.Timestamp("2026-08-01 00:00:00")

In [35]:
# Keep only transactions before the scoring date
df = df[df["ts"] < SCORING_TS].copy()

# ----------------------------
# Create time-based features
# ----------------------------

# Extract the hour of the day (0–23)
df["hour"] = df["ts"].dt.hour

# Extract the day of the week (Monday=0, Sunday=6)
df["dow"] = df["ts"].dt.dayofweek

# Extract the day of the month (1–31)
day = df["ts"].dt.day

# ----------------------------
# Create cyclical features
# ----------------------------

# Hour of the day
df["hr_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hr_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Day of the week
df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7)

# ----------------------------
# Other useful features
# ----------------------------

# Distance from the nearest payday (1st or 28th of the month)
df["payday_dist"] = np.minimum((day - 1).abs(), (day - 28).abs())

# Reduce the effect of very large transaction amounts
df["log_amt"] = np.log1p(df["amount_abs"])

# Flag whether the GPS location is missing
df["gps_missing"] = df["gps_lat"].isna().astype(int)

In [36]:
# --- 2f. Customer-level RFM / ratio aggregates as-of SCORING_TS (Table 2.2) ---
# All aggregates use only rows with ts < SCORING_TS (enforced above), so no feature


is_debit  = df["amount_signed"] < 0
by        = df.groupby("customer_id")


agg     = pd.DataFrame({
                        "recency_days":   (SCORING_TS - by["ts"].max()).dt.total_seconds() / 86400,    # R
                        "frequency":      by.size(),                                                   # F
                        "monetary_out":   df.assign(o=np.where(is_debit,  df["amount_abs"], 0)).groupby("customer_id")["o"].sum(),  # M
                        "monetary_in":    df.assign(i=np.where(~is_debit, df["amount_abs"], 0)).groupby("customer_id")["i"].sum(),
                        "n_counterparty": by["counterparty"].nunique(),                                                          # stability: distinct counterparties
                        "cashout_ratio":  df.assign(c=(df["txn_type"]=="cashout").astype(int)).groupby("customer_id")["c"].mean(),
                    })


agg["out_in_ratio"] = agg["monetary_out"] / (agg["monetary_in"] + 1.0)  # scale-free ratio
df = df.merge(agg, on="customer_id", how="left")

print("modelling table:", df.shape, "| customers:", df["customer_id"].nunique())

df[["customer_id","recency_days","frequency","monetary_out","monetary_in",
    "n_counterparty","cashout_ratio","out_in_ratio"]].head(3)

modelling table: (180000, 38) | customers: 3991


,customer_id,recency_days,frequency,monetary_out,monetary_in,n_counterparty,cashout_ratio,out_in_ratio
0,NID101089,12.508333,125,158085.0,73259.0,123,0.136000,2.157862
1,NID103658,12.170995,59,60882.0,44395.0,58,0.118644,1.371340
2,NID101584,1.488889,31,33101.0,26404.0,31,0.129032,1.253588


## 3. Hunt the leak

A two-minute correlation screen  rank features by |ρ(f, y)| and interrogate
the top of the list. 

In [37]:
# --- 3a. Correlation screen over numeric candidates 
num_candidates = ["manual_review_score", "amount_abs", "log_amt",
                  "hr_sin", "hr_cos", "dow_sin", "dow_cos", "payday_dist", "gps_missing",
                  "recency_days", "frequency", "monetary_out", "monetary_in",
                  "n_counterparty", "cashout_ratio", "out_in_ratio"]


y = df["is_fraud"].values


corr = pd.Series({c: abs(np.corrcoef(df[c].fillna(df[c].median()), y)[0, 1]) for c in num_candidates}).sort_values(ascending=False)


print("|corr| with is_fraud:")
print(corr.round(3).to_string())

# manual_review_score ~0.98 <-- statistical smoke.

|corr| with is_fraud:
manual_review_score    0.981
hr_cos                 0.071
hr_sin                 0.032
log_amt                0.022
amount_abs             0.016
cashout_ratio          0.010
monetary_out           0.004
frequency              0.003
n_counterparty         0.003
monetary_in            0.003
payday_dist            0.002
out_in_ratio           0.002
dow_sin                0.002
recency_days           0.002
gps_missing            0.000
dow_cos                0.000


In [38]:
# --- 3b. The categorical suspect: settlement_status class separation ---

print("fraud rate by settlement_status:")
print(df.groupby("settlement_status")["is_fraud"].mean().round(3).to_string())

print("\ncounts:")
print(df["settlement_status"].value_counts().to_string())
# 'reversed' -> 100% fraud, 'held' -> ~65%, 'settled'/'pending' -> ~0%: near-perfect split.

fraud rate by settlement_status:
settlement_status
held        0.651
pending     0.000
reversed    1.000
settled     0.004

counts:
settlement_status
settled     154760
reversed      9903
pending       8241
held          7096


## 4. Pipeline

One `ColumnTransformer` + `Pipeline`  so every imputer / transform / encoder
is re-fit **inside** each CV fold  preprocessing leakage becomes structurally impossible.
Evaluated with `GroupKFold` grouped by the resolved `customer_id`. We report ROC–AUC **without** and **with** the leaky columns.

In [39]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GroupKFold

NUMERICAL_COLS = ["amount_abs", "log_amt", "hr_sin", "hr_cos", "dow_sin", "dow_cos",
              "payday_dist", "gps_missing", "recency_days", "frequency",
              "monetary_out", "monetary_in", "n_counterparty", "cashout_ratio", "out_in_ratio"]

CATEGORY_COLS = ["txn_type", "region", "segment"]

LEAK_NUM   = ["manual_review_score"]     # <- to be dropped
LEAK_CAT   = ["settlement_status"]       # <- to be dropped

def build_model(num_cols, cat_cols):
    num = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("power",  PowerTransformer(method="yeo-johnson")),  # transform to Gaussian-like for LR
        ("scale",  RobustScaler()),# scaling
    ])
    cat = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="UNK")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50)),
    ])
    prep = ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)])
    
    results= Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000))]) # Pipeline with preprocessing and classifier
    
    return results

y       = df["is_fraud"].values
groups  = df["customer_id"].values           # split by resolved customer, never by SIM
cv      = GroupKFold(n_splits=5)

In [40]:
# --- Honest model: leaky columns EXCLUDED ---


clean     = build_model(NUMERICAL_COLS, CATEGORY_COLS)
auc_clean = cross_val_score(clean, df, y, cv=cv, groups=groups, scoring="roc_auc")


print(f"ROC-AUC WITHOUT leaks: {auc_clean.mean():.4f}  (+/- {auc_clean.std():.4f})")

ROC-AUC WITHOUT leaks: 0.6252  (+/- 0.0030)


In [41]:
# --- Contaminated model: leaky columns INCLUDED ---

leaky     = build_model(NUMERICAL_COLS + LEAK_NUM, CATEGORY_COLS + LEAK_CAT)
auc_leaky = cross_val_score(leaky, df, y, cv=cv, groups=groups, scoring="roc_auc")


print(f"ROC-AUC WITH leaks   : {auc_leaky.mean():.4f}  (+/- {auc_leaky.std():.4f})")
print(f"Offline AUC inflation from leakage: {auc_leaky.mean() - auc_clean.mean():+.4f}")

ROC-AUC WITH leaks   : 1.0000  (+/- 0.0000)
Offline AUC inflation from leakage: +0.3748


### Step 4 comment on the gap

Including the two post-outcome fields lifts cross-validated ROC–AUC to **~1.00**, versus
**~0.63** for the honest feature set  an inflation of roughly **+0.37 AUC**. Every
offline safeguard (CV, held-out AUC, model comparison) rewards the leak, because on the
contaminated data the leaked columns genuinely predict the label. In production those
fields are written *after* the transaction is scored, so they would be absent (or null),
and the ~1.00 model would collapse toward the honest ~0.63 — or worse, since it learned to
lean on columns it can no longer see.

Two structural safeguards made the honest number trustworthy: (1) all preprocessing lives
**inside** the `Pipeline`, re-fit per fold, so no imputer/scaler statistic crosses the
train/validation boundary; and (2) `GroupKFold` on the resolved `customer_id` keeps every
SIM of a person on one side of each split (leakage cause 5). A leaky number is worth less
than an honest one.

### Additions

In [42]:
# A. Expanded evidence for the 10-column Data Quality Audit

audit_rows = []

def add_issue(column, issue, evidence, missing_class, remediation):
    audit_rows.append({
        "Column": column,
        "Data quality issue": issue,
        "Evidence": evidence,
        "Missingness class": missing_class,
        "Remediation": remediation
    })

# 1-4: incomplete fields
for col in ["gps_lat", "gps_lon", "agent_id", "device_id"]:
    pct = raw[col].isna().mean() * 100
    add_issue(
        col,
        "Missing values",
        f"{pct:.2f}% missing ({raw[col].isna().sum():,}/{len(raw):,} rows).",
        {"gps_lat":"MNAR", "gps_lon":"MNAR", "agent_id":"MAR", "device_id":"MCAR"}[col],
        {
            "gps_lat":"Keep a missingness indicator; median-impute inside the pipeline when used.",
            "gps_lon":"Treat jointly with latitude; retain missingness information and impute only inside the pipeline.",
            "agent_id":"Missingness depends strongly on observed region; use an explicit UNK category if used.",
            "device_id":"Use an explicit UNK category if used; do not impute before cross-validation."
        }[col]
    )

# 5: duplicated transaction identifier
add_issue(
    "txn_id",
    "Duplicate transaction records",
    f"{raw['txn_id'].duplicated().sum():,} duplicate txn_id values; "
    f"{raw.duplicated().sum():,} rows are exact duplicates.",
    "N/A",
    "Drop exact duplicate rows before modelling so repeated copies of the same event are not counted twice."
)

# 6: mixed amount representations
amount_examples = raw["amount"].dropna().sample(8, random_state=1).tolist()
add_issue(
    "amount",
    "Stored as text with inconsistent debit notation",
    f"dtype={raw['amount'].dtype}; examples={amount_examples}. "
    "Debit values appear as leading '-', parentheses, or 'Dr'.",
    "N/A",
    "Parse deterministically to signed numeric amount; validate that parsing produces zero failures."
)

# 7: mixed timestamp formats
time_examples = raw["txn_time"].sample(8, random_state=2).tolist()
add_issue(
    "txn_time",
    "Mixed timestamp formats",
    f"dtype={raw['txn_time'].dtype}; examples={time_examples}. "
    "ISO, dd/mm/yyyy and month-name formats coexist.",
    "N/A",
    "Parse each known format explicitly and validate zero NaT values; enforce ts < scoring timestamp."
)

# 8: inconsistent registration ID formatting
reg_clean = raw["reg_id"].astype(str).str.strip().str.upper()
add_issue(
    "reg_id",
    "Entity identifier formatting inconsistency",
    f"{raw['reg_id'].nunique():,} raw IDs reduce to {reg_clean.nunique():,} after strip+uppercase.",
    "N/A",
    "Canonicalise with strip+uppercase and use the resulting customer_id for grouping and GroupKFold."
)

# 9: inconsistent customer-name formatting
name_raw_n = raw["account_name"].nunique()
name_clean = (raw["account_name"].astype(str)
              .str.strip().str.upper()
              .str.replace(r"\s+", " ", regex=True)
              .str.replace(",", "", regex=False))
name_clean_n = name_clean.nunique()
add_issue(
    "account_name",
    "Inconsistent casing, whitespace and name presentation",
    f"{name_raw_n:,} raw unique strings vs {name_clean_n:,} after basic case/whitespace/punctuation normalisation; "
    "examples in the notebook include uppercase, double spaces, initials and 'surname, firstname' forms.",
    "N/A",
    "Do not use the name as a model feature. Use canonical reg_id as the customer key; retain names only for operational reference."
)

# 10: post-outcome settlement field
settle_rates = raw.groupby("settlement_status")["is_fraud"].mean().round(3).to_dict()
add_issue(
    "settlement_status",
    "Availability-time / leakage defect",
    f"Fraud rate by status={settle_rates}; in the cleaned data, 'reversed' is 100% fraud and 'held' about 65%.",
    "N/A",
    "Exclude from the deployable feature set because settlement outcome is written after the scoring decision."
)

audit_table = pd.DataFrame(audit_rows)
pd.set_option("display.max_colwidth", 120)
display(audit_table)


,Column,Data quality issue,Evidence,Missingness class,Remediation
0,gps_lat,Missing values,"36.21% missing (66,483/183,600 rows).",MNAR,Keep a missingness indicator; median-impute inside the pipeline when used.
1,gps_lon,Missing values,"36.21% missing (66,483/183,600 rows).",MNAR,Treat jointly with latitude; retain missingness information and impute only inside the pipeline.
2,agent_id,Missing values,"13.69% missing (25,142/183,600 rows).",MAR,Missingness depends strongly on observed region; use an explicit UNK category if used.
3,device_id,Missing values,"5.91% missing (10,856/183,600 rows).",MCAR,Use an explicit UNK category if used; do not impute before cross-validation.
4,txn_id,Duplicate transaction records,"3,600 duplicate txn_id values; 3,600 rows are exact duplicates.",N/A,Drop exact duplicate rows before modelling so repeated copies of the same event are not counted twice.
5,amount,Stored as text with inconsistent debit notation,"dtype=object; examples=['71/- Dr', '4,742/- Dr', '1,701/-', '(227/-)', '(165/-)', '(51/-)', '-316/-', '-2,130/-']. D...",N/A,Parse deterministically to signed numeric amount; validate that parsing produces zero failures.
6,txn_time,Mixed timestamp formats,"dtype=object; examples=['29/06/2026 04:49', '2025-09-09 05:09:38', 'Mar 20, 2026 02:09 PM', 'May 04, 2025 08:43 PM',...",N/A,Parse each known format explicitly and validate zero NaT values; enforce ts < scoring timestamp.
7,reg_id,Entity identifier formatting inconsistency,"11,722 raw IDs reduce to 3,991 after strip+uppercase.",N/A,Canonicalise with strip+uppercase and use the resulting customer_id for grouping and GroupKFold.
8,account_name,"Inconsistent casing, whitespace and name presentation","1,232 raw unique strings vs 602 after basic case/whitespace/punctuation normalisation; examples in the notebook incl...",N/A,Do not use the name as a model feature. Use canonical reg_id as the customer key; retain names only for operational ...
9,settlement_status,Availability-time / leakage defect,"Fraud rate by status={'held': 0.65, 'pending': 0.0, 'reversed': 1.0, 'settled': 0.004}; in the cleaned data, 'revers...",N/A,Exclude from the deployable feature set because settlement outcome is written after the scoring decision.


In [43]:
# B. Feature dictionary aligned to Chapter 2

def window_features(data, days, scoring_ts):
    w = data[(data["ts"] >= scoring_ts - pd.Timedelta(days=days)) &
             (data["ts"] < scoring_ts)].copy()
    debit = w["amount_signed"] < 0
    g = w.groupby("customer_id")

    out = pd.DataFrame(index=df["customer_id"].drop_duplicates().sort_values())
    out[f"frequency_{days}d"] = g.size()
    out[f"monetary_out_{days}d"] = (
        w.assign(v=np.where(debit, w["amount_abs"], 0))
         .groupby("customer_id")["v"].sum()
    )
    out[f"monetary_in_{days}d"] = (
        w.assign(v=np.where(~debit, w["amount_abs"], 0))
         .groupby("customer_id")["v"].sum()
    )
    out[f"out_in_ratio_{days}d"] = (
        out[f"monetary_out_{days}d"] / (out[f"monetary_in_{days}d"] + 1.0)
    )
    out[f"cashout_ratio_{days}d"] = (
        w.assign(v=(w["txn_type"] == "cashout").astype(int))
         .groupby("customer_id")["v"].mean()
    )
    out[f"n_counterparty_{days}d"] = g["counterparty"].nunique()
    out[f"night_share_{days}d"] = (
        w.assign(v=((w["hour"] >= 22) | (w["hour"] < 5)).astype(int))
         .groupby("customer_id")["v"].mean()
    )
    return out.fillna(0)

f7 = window_features(df, 7, SCORING_TS)
f30 = window_features(df, 30, SCORING_TS)
f90 = window_features(df, 90, SCORING_TS)

assignment_features = f7.join(f30, how="outer").join(f90, how="outer").fillna(0)

# Velocity/trend: compare recent weekly activity with the prior 30-day level.
assignment_features["txn_velocity_7v30"] = (
    assignment_features["frequency_7d"] / 7.0 -
    assignment_features["frequency_30d"] / 30.0
)
assignment_features["inflow_velocity_7v30"] = (
    assignment_features["monetary_in_7d"] / 7.0 -
    assignment_features["monetary_in_30d"] / 30.0
)

# Stability: standard deviation of weekly inflows during the last 90 days.
w90 = df[(df["ts"] >= SCORING_TS - pd.Timedelta(days=90)) &
         (df["ts"] < SCORING_TS)].copy()
w90["week_bucket"] = ((SCORING_TS - w90["ts"]).dt.days // 7).astype(int)
w90["inflow"] = np.where(w90["amount_signed"] >= 0, w90["amount_abs"], 0)
weekly_in = (w90.groupby(["customer_id", "week_bucket"])["inflow"].sum()
                .groupby("customer_id").std()
                .fillna(0))
assignment_features["weekly_inflow_std_90d"] = weekly_in

feature_dictionary = pd.DataFrame([
    ["recency_days","RFM","Days since the customer's latest transaction.","All history before t",
     "txn_time -> ts; reg_id -> customer_id","Transaction timestamp and KYC ID exist before scoring.",
     "Short-history customers may differ systematically from established customers."],
    ["frequency_7d","RFM","Number of transactions in the previous 7 days.","[t-7d, t)",
     "txn_id; txn_time; customer_id","All written at transaction/KYC time.",
     "High digital activity may reflect access rather than creditworthiness."],
    ["frequency_30d","RFM","Number of transactions in the previous 30 days.","[t-30d, t)",
     "txn_id; txn_time; customer_id","All written before scoring.",
     "Same digital-access concern as frequency_7d."],
    ["frequency_90d","RFM","Number of transactions in the previous 90 days.","[t-90d, t)",
     "txn_id; txn_time; customer_id","All written before scoring.",
     "Longer history can favour established users over new customers."],
    ["monetary_in_30d","RFM","Total credit/inflow volume in the previous 30 days.","[t-30d, t)",
     "amount; txn_time; customer_id","Amount and timestamp are available at transaction posting.",
     "Can proxy income/wealth; review disparate impact."],
    ["monetary_out_30d","RFM","Total debit/outflow volume in the previous 30 days.","[t-30d, t)",
     "amount; txn_time; customer_id","Available at transaction posting.",
     "Spending volume may proxy socioeconomic position."],
    ["out_in_ratio_30d","Ratios","30-day outflows divided by 30-day inflows + 1.","[t-30d, t)",
     "monetary_out_30d; monetary_in_30d","Both aggregates use only pre-scoring transactions.",
     "Scale-free, but unstable for customers with very low inflows."],
    ["out_in_ratio_90d","Ratios","90-day outflows divided by 90-day inflows + 1.","[t-90d, t)",
     "amount; txn_time; customer_id","All upstream transactions pre-date scoring.",
     "May reflect cash-based behaviour imperfectly."],
    ["cashout_ratio_30d","Ratios","Share of 30-day transactions that are cash-outs.","[t-30d, t)",
     "txn_type; txn_time; customer_id","Transaction type is recorded when the transaction posts.",
     "Cash dependence can reflect infrastructure/geography rather than risk."],
    ["txn_velocity_7v30","Velocity / trend","Difference between daily transaction rate over 7 days and 30 days.","7d versus 30d",
     "frequency_7d; frequency_30d","Both component windows end strictly before scoring.",
     "Sudden activity changes may differ across seasonal/occupational groups."],
    ["inflow_velocity_7v30","Velocity / trend","Difference between daily inflow rate over 7 days and 30 days.","7d versus 30d",
     "monetary_in_7d; monetary_in_30d","Both component windows end before scoring.",
     "Income timing differs for salaried and informal workers."],
    ["weekly_inflow_std_90d","Stability","Standard deviation of weekly inflow totals.","[t-90d, t)",
     "amount; txn_time; customer_id","All transaction records are available before scoring.",
     "Irregular income is common in informal work and must not automatically imply unfair risk."],
    ["n_counterparty_30d","Stability","Number of distinct counterparties in the previous 30 days.","[t-30d, t)",
     "counterparty; txn_time; customer_id","Counterparty is recorded at transaction time.",
     "Network breadth may proxy occupation or business type."],
    ["n_counterparty_90d","Stability","Number of distinct counterparties in the previous 90 days.","[t-90d, t)",
     "counterparty; txn_time; customer_id","All upstream fields pre-date scoring.",
     "Same network-proxy concern; monitor by protected groups where lawful."],
    ["night_share_30d","Behavioural flags","Share of transactions occurring from 22:00 to 04:59.","[t-30d, t)",
     "txn_time; customer_id","Timestamp exists at transaction time.",
     "Chapter 2 Remark 2.3 applies: behavioural flags may proxy age, gender or neighbourhood."],
    ["night_share_90d","Behavioural flags","Share of transactions occurring from 22:00 to 04:59.","[t-90d, t)",
     "txn_time; customer_id","Timestamp exists before scoring.",
     "Requires explicit proxy-discrimination review before deployment."]
], columns=["Feature name","Family","Definition","Computation window",
            "Upstream fields / lineage","Write time","Fairness note"])

display(feature_dictionary)
print("Documented engineered features:", len(feature_dictionary))
print("Table 2.2 families represented:", feature_dictionary["Family"].nunique())


,Feature name,Family,Definition,Computation window,Upstream fields / lineage,Write time,Fairness note
0,recency_days,RFM,Days since the customer's latest transaction.,All history before t,txn_time -> ts; reg_id -> customer_id,Transaction timestamp and KYC ID exist before scoring.,Short-history customers may differ systematically from established customers.
1,frequency_7d,RFM,Number of transactions in the previous 7 days.,"[t-7d, t)",txn_id; txn_time; customer_id,All written at transaction/KYC time.,High digital activity may reflect access rather than creditworthiness.
2,frequency_30d,RFM,Number of transactions in the previous 30 days.,"[t-30d, t)",txn_id; txn_time; customer_id,All written before scoring.,Same digital-access concern as frequency_7d.
3,frequency_90d,RFM,Number of transactions in the previous 90 days.,"[t-90d, t)",txn_id; txn_time; customer_id,All written before scoring.,Longer history can favour established users over new customers.
4,monetary_in_30d,RFM,Total credit/inflow volume in the previous 30 days.,"[t-30d, t)",amount; txn_time; customer_id,Amount and timestamp are available at transaction posting.,Can proxy income/wealth; review disparate impact.
5,monetary_out_30d,RFM,Total debit/outflow volume in the previous 30 days.,"[t-30d, t)",amount; txn_time; customer_id,Available at transaction posting.,Spending volume may proxy socioeconomic position.
6,out_in_ratio_30d,Ratios,30-day outflows divided by 30-day inflows + 1.,"[t-30d, t)",monetary_out_30d; monetary_in_30d,Both aggregates use only pre-scoring transactions.,"Scale-free, but unstable for customers with very low inflows."
7,out_in_ratio_90d,Ratios,90-day outflows divided by 90-day inflows + 1.,"[t-90d, t)",amount; txn_time; customer_id,All upstream transactions pre-date scoring.,May reflect cash-based behaviour imperfectly.
8,cashout_ratio_30d,Ratios,Share of 30-day transactions that are cash-outs.,"[t-30d, t)",txn_type; txn_time; customer_id,Transaction type is recorded when the transaction posts.,Cash dependence can reflect infrastructure/geography rather than risk.
9,txn_velocity_7v30,Velocity / trend,Difference between daily transaction rate over 7 days and 30 days.,7d versus 30d,frequency_7d; frequency_30d,Both component windows end strictly before scoring.,Sudden activity changes may differ across seasonal/occupational groups.


Documented engineered features: 16
Table 2.2 families represented: 5


## Leakage assessment summary

Chapter 2, Section 2.7.2 lists six common causes of leakage. The two intentionally planted variables in this lab are both **Cause 6: leakage from the data-collection process** because they are written downstream of the intended scoring event.

1. **`manual_review_score`**  a downstream manual-review signal. Its absolute correlation with `is_fraud` is approximately **0.981**. A model scored at transaction initiation cannot legitimately rely on a review score produced later in the investigation workflow.
2. **`settlement_status`**  a post-transaction outcome. `reversed` transactions have a fraud rate of **1.000** and `held` transactions approximately **0.651**, making it a near-direct label proxy.

Together they raise five-fold group-aware CV ROC-AUC from **0.6252 ± 0.0030** to **1.0000 ± 0.0000**, an inflation of **+0.3748**.

The notebook also prevents **Cause 2 (scaling before splitting)** and **Cause 3 (imputing with full-data statistics)** by fitting those operations inside the Scikit-learn `Pipeline`, and addresses **Cause 4/5 (duplicates/group leakage)** through deduplication and `GroupKFold` on the resolved `customer_id`.

Governance recommendation: require every feature to have documented source, owner, lineage, write time and an `available_at_scoring` flag, and reject any feature whose write time is after the scoring timestamp.


In [44]:
# C. Final reproducible model fit + persistence
# IMPORTANT: this is the honest pipeline; the two planted leakage columns are excluded.

import joblib
import sklearn
import pandas as pd
import numpy as np

FINAL_FEATURES = NUMERICAL_COLS + CATEGORY_COLS

final_pipeline = build_model(NUMERICAL_COLS, CATEGORY_COLS)
final_pipeline.fit(df[FINAL_FEATURES], y)

MODEL_PATH = "credit_risk_fraud_pipeline.joblib"
joblib.dump(final_pipeline, MODEL_PATH)

# Reload test: proves the artifact executes without rebuilding preprocessing.
reloaded_pipeline = joblib.load(MODEL_PATH)
sample_X = df[FINAL_FEATURES].head(10)
sample_proba = reloaded_pipeline.predict_proba(sample_X)[:, 1]

print("Saved:", MODEL_PATH)
print("scikit-learn version:", sklearn.__version__)
print("Reload successful:", len(sample_proba) == len(sample_X))
print("Sample probabilities:", np.round(sample_proba, 4))

# Save lightweight reproducibility metadata alongside the model.
metadata = {
    "scoring_timestamp": str(SCORING_TS),
    "cv_strategy": "GroupKFold(n_splits=5), grouped by canonical customer_id",
    "roc_auc_mean": float(auc_clean.mean()),
    "roc_auc_std": float(auc_clean.std()),
    "excluded_leakage_fields": ["manual_review_score", "settlement_status"],
    "feature_columns": FINAL_FEATURES,
    "sklearn_version": sklearn.__version__,
}
joblib.dump(metadata, "credit_risk_fraud_pipeline_metadata.joblib")
print(metadata)


Saved: credit_risk_fraud_pipeline.joblib
scikit-learn version: 1.6.1
Reload successful: True
Sample probabilities: [0.066  0.0703 0.0804 0.0703 0.1033 0.0663 0.0846 0.0458 0.0781 0.07  ]
{'scoring_timestamp': '2026-08-01 00:00:00', 'cv_strategy': 'GroupKFold(n_splits=5), grouped by canonical customer_id', 'roc_auc_mean': 0.6251632166014376, 'roc_auc_std': 0.003042435065781064, 'excluded_leakage_fields': ['manual_review_score', 'settlement_status'], 'feature_columns': ['amount_abs', 'log_amt', 'hr_sin', 'hr_cos', 'dow_sin', 'dow_cos', 'payday_dist', 'gps_missing', 'recency_days', 'frequency', 'monetary_out', 'monetary_in', 'n_counterparty', 'cashout_ratio', 'out_in_ratio', 'txn_type', 'region', 'segment'], 'sklearn_version': '1.6.1'}
